In [7]:
import numpy as np
import pandas as pd
import tensorflow
import keras
import nltk
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from scipy.sparse import triu
!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip -q glove.6B.zip
from scipy import spatial
import networkx as nx

--2026-05-23 09:43:45--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2026-05-23 09:43:46--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2026-05-23 09:43:46--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glov

In [8]:
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [9]:
text = """A sample text paragraph is a written section that illustrates a specific idea or theme. It typically includes a topic sentence, supporting details, and a concluding sentence. Sample paragraphs can be used for various purposes, such as demonstrating writing style, presenting information, or providing examples in educational contexts.

Structure of a Sample Text Paragraph
Topic Sentence: This is the first sentence that states the main idea of the paragraph. It sets the tone and direction for the rest of the text.
Supporting Sentences: These sentences provide evidence, examples, or explanations that bolster the main idea. They are essential for developing the paragraph’s theme and providing clarity.
Concluding Sentence: This wraps up the paragraph, reinforcing the main idea and providing a transition to the next point or paragraph.
Importance of Sample Text Paragraphs
Sample text paragraphs are vital for several reasons:

Clarity: Well-structured paragraphs help convey ideas clearly, making it easier for readers to understand the content.
Organization: They provide a framework for organizing thoughts, which is essential for effective communication.
Engagement: Engaging paragraphs capture the reader’s interest, encouraging them to continue reading."""

In [10]:
sentences = nltk.sent_tokenize(text)


clean_sentences = [re.sub(r'[^\w\s]',"",sentence.lower()) for sentence in sentences]


In [11]:
stopwords = stopwords.words("english")

clean_sentences = [[words for words in sentence.split()if words not in stopwords]for sentence in clean_sentences]


In [12]:
glove_embeddings = {}

with open("glove.6B.100d.txt", encoding='utf8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype='float32')
        glove_embeddings[word] = vector

In [20]:
sentence_embedding = []

for sentence in clean_sentences:
    vectors = []
    for word in sentence:
        if word in glove_embeddings:
            vectors.append(glove_embeddings[word])
    if len(vectors) != 0:
        sentence_vector = np.mean(vectors, axis=0)
    else:
        sentence_vector = np.zeros((100,))
    sentence_embedding.append(sentence_vector)

In [21]:
similarity_matrix = np.zeros((len(sentences),len(sentences)))
for i, row in enumerate(sentence_embedding):
  for j, column in enumerate(sentence_embedding):
    similarity_matrix[i][j] = 1-spatial.distance.cosine(row,column)
    if similarity_matrix[i][j] < 0:
      similarity_matrix[i][j] = 0


In [22]:
nx_graph = nx.from_numpy_array(similarity_matrix)
scores = nx.pagerank(nx_graph,max_iter=1000)


In [23]:
top_sentence = {sentence:scores[index] for index, sentence in enumerate(sentences)}


top_four = dict(sorted(top_sentence.items(),key=lambda x:x[1], reverse=True)[:4])


In [24]:
for sentence in sentences:
  if sentence in top_four.keys():
    print(sentence)

A sample text paragraph is a written section that illustrates a specific idea or theme.
Structure of a Sample Text Paragraph
Topic Sentence: This is the first sentence that states the main idea of the paragraph.
Concluding Sentence: This wraps up the paragraph, reinforcing the main idea and providing a transition to the next point or paragraph.
Importance of Sample Text Paragraphs
Sample text paragraphs are vital for several reasons:

Clarity: Well-structured paragraphs help convey ideas clearly, making it easier for readers to understand the content.
